# Fair-Explainable Clustering: from Fairness to Explainability
**Thesis Pipeline — Harish Sharma**

---
### Pipeline Overview
1. Configuration & Imports
2. Data Loading & Preprocessing
3. Fairlet Decomposition
4. Fairlet Medoid Computation
5. K-Medoids (PAM)
6. K-Medians
7. Label Assignment from Fairlets
8. Fairness Metrics
9. Explainability Metrics
10. Unified Evaluation Function
11. **Baseline: K-Means** → Metrics
12. **Baseline: K-Medians** → Metrics
13. **Fairlets + K-Medians** → Metrics
14. Explainability Analysis

---
## Cell 1 — Configuration & Imports

In [1]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.cluster import KMeans
from scipy.spatial.distance import cdist

# ============================================================
# CONFIG
# ============================================================
K_CLUSTERS   = 5
P            = 1
Q            = 1
RANDOM_STATE = 42
SAMPLE_SIZE  = 3000   # set to None to use full dataset

DATA_PATH = r'D:\Thesis\Fair_explainable_cluster_updated_2nd_feb\data\bank-full.csv'

print("Imports and config loaded successfully.")

Imports and config loaded successfully.


---
## Cell 2 — Data Loading & Preprocessing

In [2]:
def load_data():
    """
    Load the Bank Marketing dataset.
    Sensitive attribute: married=1, others=0.
    Returns preprocessed feature matrix X, sensitive array, and feature names.
    """
    df = pd.read_csv(DATA_PATH)

    # Sensitive attribute: married = 1, others = 0
    df['sensitive'] = df['marital'].apply(lambda x: 1 if x == 'married' else 0)

    drop_cols = ['marital', 'sensitive', 'y']
    X_raw     = df.drop(columns=drop_cols)
    sensitive = df['sensitive'].values

    # Optional sampling for speed
    if SAMPLE_SIZE is not None and SAMPLE_SIZE < len(X_raw):
        idx       = np.random.choice(len(X_raw), SAMPLE_SIZE, replace=False)
        X_raw     = X_raw.iloc[idx].reset_index(drop=True)
        sensitive = sensitive[idx]

    # Separate numeric and categorical columns
    categorical_cols = X_raw.select_dtypes(include=['object']).columns
    numeric_cols     = X_raw.select_dtypes(include=['number']).columns

    preprocessor = ColumnTransformer([
        ('num', StandardScaler(),                   numeric_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ])

    X_processed   = preprocessor.fit_transform(X_raw)
    feature_names = list(preprocessor.get_feature_names_out())

    return X_processed, sensitive, feature_names

In [3]:
# Run data loading
X, sensitive, feature_names = load_data()

print(f"Dataset shape      : {X.shape}")
print(f"Sensitive (married): {sensitive.sum()} / {len(sensitive)} "
      f"({sensitive.mean()*100:.1f}%)")
print(f"Number of features : {len(feature_names)}")

Dataset shape      : (3000, 48)
Sensitive (married): 2045 / 3000 (68.2%)
Number of features : 48


C:\Users\91773\AppData\Local\Temp\ipykernel_21308\2349511443.py:23: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X_raw.select_dtypes(include=['object']).columns


---
## Cell 3 — Fairlet Decomposition

In [4]:
def fairlet_decomposition(X, sensitive, p=1, q=1):
    """
    Decompose data into fairlets — small balanced micro-groups.
    Each fairlet contains q 'reds' (sensitive=1) and p 'blues' (sensitive=0).

    Parameters
    ----------
    X         : feature matrix (used implicitly via indices)
    sensitive : binary array of group membership
    p, q      : balance parameters (default 1:1)

    Returns
    -------
    fairlets  : list of index lists
    """
    reds  = np.where(sensitive == 1)[0].tolist()   # married
    blues = np.where(sensitive == 0)[0].tolist()   # others

    fairlets = []

    # Pair full q-reds with p-blues
    while len(reds) >= q and len(blues) >= p:
        r = [reds.pop()  for _ in range(q)]
        b = [blues.pop() for _ in range(p)]
        fairlets.append(r + b)

    # Handle remaining singles
    while len(reds) > 0 and len(blues) > 0:
        fairlets.append([reds.pop(), blues.pop()])

    return fairlets

In [5]:
# Run fairlet decomposition
fairlets = fairlet_decomposition(X, sensitive, p=P, q=Q)

sizes = [len(f) for f in fairlets]
print(f"Number of fairlets : {len(fairlets)}")
print(f"Fairlet size range : {min(sizes)} – {max(sizes)}")
print(f"Avg fairlet size   : {np.mean(sizes):.2f}")

Number of fairlets : 955
Fairlet size range : 2 – 2
Avg fairlet size   : 2.00


---
## Cell 4 — Fairlet Medoid Computation

In [6]:
def compute_fairlet_centers(X, fairlets):
    """
    For each fairlet, find the medoid (point minimising total intra-fairlet distance).

    Returns
    -------
    centers : array of original data indices that are fairlet medoids
    """
    centers = []

    for fl in fairlets:
        pts          = X[fl]
        D            = cdist(pts, pts)
        medoid_index = np.argmin(D.sum(axis=1))
        centers.append(fl[medoid_index])

    return np.array(centers)

In [7]:
# Compute fairlet medoids
fairlet_center_indices = compute_fairlet_centers(X, fairlets)

print(f"Fairlet centers computed: {len(fairlet_center_indices)} medoids")
print(f"Sample indices          : {fairlet_center_indices[:10]}")

Fairlet centers computed: 955 medoids
Sample indices          : [2999 2998 2997 2996 2995 2994 2992 2991 2988 2987]


---
## Cell 5 — K-Medoids (PAM)

In [8]:
# def kmedoids(X, k, max_iter=100, random_state=42):
#     """
#     K-Medoids clustering using the PAM (Partitioning Around Medoids) algorithm.
#     Uses Euclidean distance.

#     Parameters
#     ----------
#     X           : feature matrix
#     k           : number of clusters
#     max_iter    : maximum iterations
#     random_state: random seed

#     Returns
#     -------
#     labels         : cluster assignment for each point
#     medoid_indices : indices of final medoids
#     """
#     np.random.seed(random_state)
#     n = X.shape[0]

#     medoid_indices = np.random.choice(n, k, replace=False)

#     for _ in range(max_iter):
#         distances = cdist(X, X[medoid_indices])
#         labels    = np.argmin(distances, axis=1)

#         new_medoids = []
#         for i in range(k):
#             cluster_points = np.where(labels == i)[0]
#             if len(cluster_points) == 0:
#                 new_medoids.append(medoid_indices[i])  # keep old medoid
#                 continue
#             cluster_distances = cdist(X[cluster_points], X[cluster_points])
#             best_medoid       = cluster_points[np.argmin(cluster_distances.sum(axis=1))]
#             new_medoids.append(best_medoid)

#         new_medoids = np.array(new_medoids)
#         if np.all(new_medoids == medoid_indices):
#             break
#         medoid_indices = new_medoids

#     return labels, medoid_indices

---
## Cell 6 — K-Medians

In [9]:
def kmedians(X, k, max_iter=100, random_state=42):
    """
    K-Medians clustering.
    Uses L1 (Manhattan) distance for assignment; updates centres with the median.

    Parameters
    ----------
    X           : feature matrix
    k           : number of clusters
    max_iter    : maximum iterations
    random_state: random seed

    Returns
    -------
    labels  : cluster assignment for each point
    centers : final median centres (not actual data points)
    """
    np.random.seed(random_state)
    n_samples = X.shape[0]

    # Initialise centres from random data points
    indices = np.random.choice(n_samples, k, replace=False)
    centers = X[indices]

    for _ in range(max_iter):
        # Assign using Manhattan distance
        distances = cdist(X, centers, metric='cityblock')
        labels    = np.argmin(distances, axis=1)

        new_centers = []
        for i in range(k):
            cluster_points = X[labels == i]
            if len(cluster_points) == 0:
                new_centers.append(centers[i])         # keep old centre
            else:
                new_centers.append(np.median(cluster_points, axis=0))

        new_centers = np.array(new_centers)
        if np.allclose(new_centers, centers):
            break
        centers = new_centers

    return labels, centers

---
## Cell 7 — Label Assignment from Fairlets

In [10]:
def assign_labels_from_fairlets(fairlets, center_labels, n):
    """
    Propagate the cluster label of each fairlet's medoid back to all points
    in that fairlet.

    Parameters
    ----------
    fairlets      : list of index lists
    center_labels : cluster label assigned to each fairlet centre
    n             : total number of data points

    Returns
    -------
    labels : full-length cluster assignment array
    """
    labels = np.zeros(n, dtype=int)
    for fl, lab in zip(fairlets, center_labels):
        for idx in fl:
            labels[idx] = lab
    return labels

---
## Cell 8 — Fairness Metrics

In [11]:
def fairness_metrics(labels, sensitive, r=None, b=None):
    """
    Compute fairness metrics aligned with Chierichetti et al. (2017) /
    Backurs et al. (2019).

    Paper definition:
        balance(C) = min(|C_r|/|C_b|, |C_b|/|C_r|)  in [0, 1]
        A clustering is (r,b)-fair iff balance(C) >= b/r for every cluster C.

    Metrics
    -------
    min_balance    : min balance across all clusters (worst-case cluster).
                     This IS the paper's primary fairness measure — higher = fairer.
                     After fairlet preprocessing this should be >= b/r.
    avg_balance    : mean balance across clusters (overall picture).
    violation_rate : fraction of clusters where balance < b/r threshold.
                     With P=Q=1 → b/r = 1.0 (perfect balance required).
                     Use r=2, b=1 for a looser 0.5 threshold.
    avg_dp_gap     : mean |cluster_ratio - global_ratio| (demographic parity gap).

    Parameters
    ----------
    r, b : fairness parameters; defaults to global Q (majority) and P (minority).
    """
    if r is None: r = Q   # majority colour ratio  (Q in config)
    if b is None: b = P   # minority colour ratio  (P in config)
    fairness_threshold = b / r   # paper: balance(C) >= b/r

    K            = len(np.unique(labels))
    balances     = []
    violations   = 0
    global_ratio = sensitive.mean()
    dp_gaps      = []

    for k in range(K):
        mask   = labels == k
        group  = sensitive[mask]

        n_red  = np.sum(group == 1)   # sensitive=1  (married / red)
        n_blue = np.sum(group == 0)   # sensitive=0  (others  / blue)

        if max(n_red, n_blue) == 0:
            continue

        # Paper formula: balance(C) = min(|C_r|/|C_b|, |C_b|/|C_r|)
        if min(n_red, n_blue) == 0:
            balance = 0.0   # one group completely absent
        else:
            balance = min(n_red / n_blue, n_blue / n_red)

        balances.append(balance)

        if balance < fairness_threshold:
            violations += 1

        cluster_ratio = group.mean()
        dp_gaps.append(abs(cluster_ratio - global_ratio))

    return {
        "min_balance"    : np.min(balances),    # paper's primary fairness metric
        "avg_balance"    : np.mean(balances),   # complementary overview
        "violation_rate" : violations / K,      # fraction of (r,b)-unfair clusters
        "avg_dp_gap"     : np.mean(dp_gaps)     # demographic parity gap
    }

---
## Cell 9 — Explainability Metrics (Decision Tree)

In [12]:
def explainability_metrics(X, labels, feature_names, print_rules=True):
    """
    Fit a shallow decision tree to approximate the clustering and
    report rule-based explainability metrics.

    Metrics
    -------
    tree_fidelity : accuracy of the decision tree in reproducing cluster labels
    tree_depth    : depth of the fitted tree
    tree_leaves   : number of leaf nodes

    Parameters
    ----------
    X             : feature matrix
    labels        : cluster assignments
    feature_names : list of feature name strings
    print_rules   : whether to print the extracted decision rules

    Returns
    -------
    dict of metric name -> value
    """
    clf = DecisionTreeClassifier(max_depth=4, random_state=RANDOM_STATE)
    clf.fit(X, labels)

    if print_rules:
        rules = export_text(clf, feature_names=feature_names)
        print("\n--- DECISION TREE RULES ---")
        print(rules)

    return {
        "tree_fidelity" : clf.score(X, labels),
        "tree_depth"    : clf.get_depth(),
        "tree_leaves"   : clf.get_n_leaves()
    }

---
## Cell 10 — Unified Evaluation Function

In [13]:
def evaluate_model(name, X, labels, sensitive, feature_names,
                   print_rules=False):
    """
    Print clustering quality, fairness, and explainability metrics for a model.

    Parameters
    ----------
    name          : model label (shown in header)
    X             : feature matrix
    labels        : cluster assignments
    sensitive     : sensitive attribute array
    feature_names : feature name list
    print_rules   : pass True to print decision tree rules inline

    Returns
    -------
    results dict with all metric values
    """
    print(f"\n{'='*50}")
    print(f"  RESULTS: {name}")
    print(f"{'='*50}")

    sil = silhouette_score(X, labels)
    db  = davies_bouldin_score(X, labels)
    print(f"\n[CLUSTERING QUALITY]")
    print(f"  Silhouette Score   : {sil:.4f}")
    print(f"  Davies-Bouldin Idx : {db:.4f}")

    fm = fairness_metrics(labels, sensitive)
    print(f"\n[FAIRNESS]")
    for k, v in fm.items():
        print(f"  {k:<20}: {v:.4f}")

    em = explainability_metrics(X, labels, feature_names,
                                 print_rules=print_rules)
    print(f"\n[EXPLAINABILITY (Rule-Based)]")
    for k, v in em.items():
        print(f"  {k:<20}: {v}")

    return {"model": name,
            "silhouette": sil, "davies_bouldin": db,
            **fm, **em}

---
## Cell 11 — Baseline: K-Means → Metrics

In [14]:
# ── Baseline 1: K-Means ──────────────────────────────────────
kmeans = KMeans(
    n_clusters=K_CLUSTERS,
    random_state=RANDOM_STATE,
    n_init=10
)
kmeans_labels = kmeans.fit_predict(X)

results_kmeans = evaluate_model(
    "BASELINE: K-Means",
    X, kmeans_labels, sensitive, feature_names
)


  RESULTS: BASELINE: K-Means

[CLUSTERING QUALITY]
  Silhouette Score   : 0.1224
  Davies-Bouldin Idx : 2.0143

[FAIRNESS]
  min_balance         : 0.1081
  avg_balance         : 0.4921
  violation_rate      : 1.0000
  avg_dp_gap          : 0.0842

[EXPLAINABILITY (Rule-Based)]
  tree_fidelity       : 0.936
  tree_depth          : 4
  tree_leaves         : 16


---
## Cell 12 — Baseline: K-Medians → Metrics

In [15]:
# ── Baseline 2: K-Medians ────────────────────────────────────
kmedians_labels, kmedians_centers = kmedians(
    X,
    K_CLUSTERS,
    random_state=RANDOM_STATE
)

results_kmedians = evaluate_model(
    "BASELINE: K-Medians",
    X, kmedians_labels, sensitive, feature_names
)


  RESULTS: BASELINE: K-Medians

[CLUSTERING QUALITY]
  Silhouette Score   : 0.0403
  Davies-Bouldin Idx : 3.5442

[FAIRNESS]
  min_balance         : 0.2978
  avg_balance         : 0.4999
  violation_rate      : 1.0000
  avg_dp_gap          : 0.0617

[EXPLAINABILITY (Rule-Based)]
  tree_fidelity       : 0.7886666666666666
  tree_depth          : 4
  tree_leaves         : 16


---
## Cell 13 — Fairlets + K-Medians → Metrics

In [16]:
# ── Proposed: Fairlets + K-Medians ───────────────────────────

# Step 1: Build fairlets
fairlets = fairlet_decomposition(X, sensitive, p=P, q=Q)

# Step 2: Find medoid of each fairlet → reduce to one point per fairlet
fairlet_center_indices = compute_fairlet_centers(X, fairlets)

# Step 3: Cluster the fairlet medoids with K-Medians
center_labels, _ = kmedians(
    X[fairlet_center_indices],
    K_CLUSTERS,
    random_state=RANDOM_STATE
)

# Step 4: Propagate labels back to all points
fair_labels = assign_labels_from_fairlets(
    fairlets,
    center_labels,
    n=len(X)
)

results_fair = evaluate_model(
    "FAIRLETS + K-Medians",
    X, fair_labels, sensitive, feature_names
)


  RESULTS: FAIRLETS + K-Medians

[CLUSTERING QUALITY]
  Silhouette Score   : -0.0375
  Davies-Bouldin Idx : 8.2695

[FAIRNESS]
  min_balance         : 0.1058
  avg_balance         : 0.8212
  violation_rate      : 0.2000
  avg_dp_gap          : 0.1899

[EXPLAINABILITY (Rule-Based)]
  tree_fidelity       : 0.4653333333333333
  tree_depth          : 4
  tree_leaves         : 16


---
## Cell 14 — Explainability Analysis (with Decision Rules printed)

In [17]:
# ── Decision Tree Rules: K-Means ─────────────────────────────
print("=" * 50)
print("  DECISION TREE RULES: K-Means")
print("=" * 50)
_ = explainability_metrics(X, kmeans_labels, feature_names, print_rules=True)

  DECISION TREE RULES: K-Means

--- DECISION TREE RULES ---
|--- num__age <= 0.22
|   |--- num__pdays <= 0.49
|   |   |--- num__duration <= 1.22
|   |   |   |--- num__campaign <= 1.72
|   |   |   |   |--- class: 3
|   |   |   |--- num__campaign >  1.72
|   |   |   |   |--- class: 2
|   |   |--- num__duration >  1.22
|   |   |   |--- num__duration <= 1.32
|   |   |   |   |--- class: 0
|   |   |   |--- num__duration >  1.32
|   |   |   |   |--- class: 0
|   |--- num__pdays >  0.49
|   |   |--- num__pdays <= 0.93
|   |   |   |--- num__previous <= 0.55
|   |   |   |   |--- class: 3
|   |   |   |--- num__previous >  0.55
|   |   |   |   |--- class: 4
|   |   |--- num__pdays >  0.93
|   |   |   |--- num__duration <= 2.58
|   |   |   |   |--- class: 4
|   |   |   |--- num__duration >  2.58
|   |   |   |   |--- class: 0
|--- num__age >  0.22
|   |--- num__pdays <= 0.66
|   |   |--- num__duration <= 1.09
|   |   |   |--- num__campaign <= 1.72
|   |   |   |   |--- class: 1
|   |   |   |--- num__

In [18]:
# ── Decision Tree Rules: K-Medians ───────────────────────────
print("=" * 50)
print("  DECISION TREE RULES: K-Medians")
print("=" * 50)
_ = explainability_metrics(X, kmedians_labels, feature_names, print_rules=True)

  DECISION TREE RULES: K-Medians

--- DECISION TREE RULES ---
|--- cat__contact_unknown <= 0.50
|   |--- cat__education_tertiary <= 0.50
|   |   |--- num__day <= -0.51
|   |   |   |--- cat__loan_no <= 0.50
|   |   |   |   |--- class: 2
|   |   |   |--- cat__loan_no >  0.50
|   |   |   |   |--- class: 0
|   |   |--- num__day >  -0.51
|   |   |   |--- cat__housing_no <= 0.50
|   |   |   |   |--- class: 1
|   |   |   |--- cat__housing_no >  0.50
|   |   |   |   |--- class: 3
|   |--- cat__education_tertiary >  0.50
|   |   |--- cat__housing_no <= 0.50
|   |   |   |--- num__day <= -0.16
|   |   |   |   |--- class: 0
|   |   |   |--- num__day >  -0.16
|   |   |   |   |--- class: 3
|   |   |--- cat__housing_no >  0.50
|   |   |   |--- cat__loan_no <= 0.50
|   |   |   |   |--- class: 2
|   |   |   |--- cat__loan_no >  0.50
|   |   |   |   |--- class: 3
|--- cat__contact_unknown >  0.50
|   |--- cat__housing_yes <= 0.50
|   |   |--- cat__education_secondary <= 0.50
|   |   |   |--- cat__loan_y

In [19]:
# ── Decision Tree Rules: Fairlets + K-Medians ─────────────────
print("=" * 50)
print("  DECISION TREE RULES: Fairlets + K-Medians")
print("=" * 50)
_ = explainability_metrics(X, fair_labels, feature_names, print_rules=True)

  DECISION TREE RULES: Fairlets + K-Medians

--- DECISION TREE RULES ---
|--- cat__housing_no <= 0.50
|   |--- cat__contact_unknown <= 0.50
|   |   |--- num__pdays <= 0.42
|   |   |   |--- num__duration <= -0.71
|   |   |   |   |--- class: 0
|   |   |   |--- num__duration >  -0.71
|   |   |   |   |--- class: 0
|   |   |--- num__pdays >  0.42
|   |   |   |--- num__age <= 0.22
|   |   |   |   |--- class: 0
|   |   |   |--- num__age >  0.22
|   |   |   |   |--- class: 0
|   |--- cat__contact_unknown >  0.50
|   |   |--- num__age <= -0.73
|   |   |   |--- num__balance <= -0.62
|   |   |   |   |--- class: 0
|   |   |   |--- num__balance >  -0.62
|   |   |   |   |--- class: 4
|   |   |--- num__age >  -0.73
|   |   |   |--- cat__job_services <= 0.50
|   |   |   |   |--- class: 0
|   |   |   |--- cat__job_services >  0.50
|   |   |   |   |--- class: 0
|--- cat__housing_no >  0.50
|   |--- cat__education_secondary <= 0.50
|   |   |--- num__age <= -0.63
|   |   |   |--- num__duration <= -0.71
| 

---
## Cell 15 — Summary Comparison Table

In [20]:
# ── Side-by-side comparison ───────────────────────────────────
summary = pd.DataFrame([
    results_kmeans,
    results_kmedians,
    results_fair
]).set_index('model')

# Round for readability
summary = summary.round(4)

print("\n=== FULL COMPARISON ===")
print(summary.T.to_string())
summary


=== FULL COMPARISON ===
model           BASELINE: K-Means  BASELINE: K-Medians  FAIRLETS + K-Medians
silhouette                 0.1224               0.0403               -0.0375
davies_bouldin             2.0143               3.5442                8.2695
min_balance                0.1081               0.2978                0.1058
avg_balance                0.4921               0.4999                0.8212
violation_rate             1.0000               1.0000                0.2000
avg_dp_gap                 0.0842               0.0617                0.1899
tree_fidelity              0.9360               0.7887                0.4653
tree_depth                 4.0000               4.0000                4.0000
tree_leaves               16.0000              16.0000               16.0000


,silhouette,davies_bouldin,min_balance,avg_balance,violation_rate,avg_dp_gap,tree_fidelity,tree_depth,tree_leaves
model,,,,,,,,,
BASELINE: K-Means,0.1224,2.0143,0.1081,0.4921,1.0,0.0842,0.9360,4,16
BASELINE: K-Medians,0.0403,3.5442,0.2978,0.4999,1.0,0.0617,0.7887,4,16
FAIRLETS + K-Medians,-0.0375,8.2695,0.1058,0.8212,0.2,0.1899,0.4653,4,16
